In [19]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle


# =========================
# USER CONFIG
# =========================

OUTPUT_FORMAT = "webm"   # "webm", "mp4"

LIGHT_MODE = "random"
# "random"
# "bottom_up"
# "top_down"

DURATION_SEC = 12
FPS = 30

LABELS = [
    "RADAR",
    "GENERATOR",
    "ENGINES",
    "STEERING",
    "FIRE CTRL",
    "PUMPS",
]


# =========================
# PATH CONFIG
# =========================

BACKGROUND_PATH = Path("assets/Binary_sea.png")

OUT_DIR = Path("animations/binary_sea")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"binary_sea_panel.{OUTPUT_FORMAT}"


# =========================
# LAYOUT CONFIG
# =========================

DPI = 140

FONT_SIZE = 8
FONT_FAMILY = "DejaVu Sans Mono"

TEXT_X_NORM = 0.18

INDICATOR_RADIUS_PX = 7
GLOW_RADIUS_1_MULT = 1.9
GLOW_RADIUS_2_MULT = 3.1

# Точные центры индикаторов, сверху вниз.
# Формат: (x_norm, y_norm), где 0..1 — доля ширины/высоты битмапа.
INDICATOR_CENTERS_NORM = [
    (0.700, 0.180),  # RADAR
    (0.705, 0.305),  # GENERATOR
    (0.702, 0.430),  # ENGINES
    (0.700, 0.537),  # STEERING
    (0.705, 0.675),  # FIRE CTRL
    (0.700, 0.820),  # PUMPS
]


# =========================
# VISUAL CONFIG
# =========================

TEXT_COLOR_OFF = "#2E7D82"
TEXT_COLOR_ON = "#6FFFF5"

INDICATOR_COLOR = "#35FFF3"
INDICATOR_CORE_COLOR = "#D8FFFB"

RANDOM_SEED = 42

STATE_HOLD_FRAMES = 18
FADE_SPEED = 0.18


# =========================
# DERIVED CONFIG
# =========================

N_FRAMES = DURATION_SEC * FPS

GLOW_RADIUS_1_PX = INDICATOR_RADIUS_PX * GLOW_RADIUS_1_MULT
GLOW_RADIUS_2_PX = INDICATOR_RADIUS_PX * GLOW_RADIUS_2_MULT


# =========================
# LOAD BACKGROUND
# =========================

if not BACKGROUND_PATH.exists():
    raise FileNotFoundError(f"Background not found: {BACKGROUND_PATH}")

img = Image.open(BACKGROUND_PATH).convert("RGB")
W, H = img.size

if len(LABELS) != len(INDICATOR_CENTERS_NORM):
    raise ValueError("LABELS and INDICATOR_CENTERS_NORM must have the same length")

N = len(LABELS)

indicator_centers = np.array(
    [(x * W, y * H) for x, y in INDICATOR_CENTERS_NORM],
    dtype=float,
)

text_x = TEXT_X_NORM * W
text_y = indicator_centers[:, 1]


# =========================
# FIGURE
# =========================

fig_w = W / DPI
fig_h = H / DPI

fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=DPI)
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

ax.imshow(img, extent=[0, W, H, 0])
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.axis("off")


# =========================
# ARTISTS
# =========================

glow_outer = []
glow_inner = []
cores = []
texts = []

for label, (cx, cy), ty in zip(LABELS, indicator_centers, text_y):
    g2 = Circle(
        (cx, cy),
        radius=GLOW_RADIUS_2_PX,
        facecolor=INDICATOR_COLOR,
        edgecolor="none",
        alpha=0.0,
    )
    ax.add_patch(g2)
    glow_outer.append(g2)

    g1 = Circle(
        (cx, cy),
        radius=GLOW_RADIUS_1_PX,
        facecolor=INDICATOR_COLOR,
        edgecolor="none",
        alpha=0.0,
    )
    ax.add_patch(g1)
    glow_inner.append(g1)

    core = Circle(
        (cx, cy),
        radius=INDICATOR_RADIUS_PX,
        facecolor=INDICATOR_CORE_COLOR,
        edgecolor=INDICATOR_COLOR,
        linewidth=0.8,
        alpha=0.0,
    )
    ax.add_patch(core)
    cores.append(core)

    text = ax.text(
        text_x,
        ty,
        label,
        color=TEXT_COLOR_OFF,
        fontsize=FONT_SIZE,
        fontfamily=FONT_FAMILY,
        fontweight="bold",
        ha="left",
        va="center",
        alpha=0.75,
    )
    texts.append(text)


# =========================
# SWITCHING LOGIC
# =========================

rng = np.random.default_rng(RANDOM_SEED)

states = np.zeros(N, dtype=float)
targets = np.zeros(N, dtype=float)

if LIGHT_MODE == "bottom_up":
    order = list(reversed(range(N)))

elif LIGHT_MODE == "top_down":
    order = list(range(N))

elif LIGHT_MODE == "random":
    order = list(rng.permutation(N))

else:
    raise ValueError(f"Unsupported LIGHT_MODE: {LIGHT_MODE}")


def get_switch_index(step: int) -> int:
    global order

    if LIGHT_MODE == "random":
        if step % N == 0:
            order = list(rng.permutation(N))
        return order[step % N]

    return order[step % N]


# =========================
# ANIMATION
# =========================

def update(frame_idx):
    global targets, states

    if frame_idx % STATE_HOLD_FRAMES == 0:
        step = frame_idx // STATE_HOLD_FRAMES
        idx = get_switch_index(step)
        targets[idx] = 1.0 - targets[idx]

    states += (targets - states) * FADE_SPEED

    pulse = 0.75 + 0.25 * np.sin(2 * np.pi * frame_idx / FPS * 2.0)

    for i in range(N):
        a = states[i]

        glow_outer[i].set_alpha(0.16 * a * pulse)
        glow_inner[i].set_alpha(0.34 * a * pulse)
        cores[i].set_alpha(0.95 * a)

        if a > 0.05:
            texts[i].set_color(TEXT_COLOR_ON)
            texts[i].set_alpha(0.50 + 0.45 * a)
        else:
            texts[i].set_color(TEXT_COLOR_OFF)
            texts[i].set_alpha(0.70)

    return [*glow_outer, *glow_inner, *cores, *texts]


anim = FuncAnimation(
    fig,
    update,
    frames=N_FRAMES,
    interval=1000 / FPS,
    blit=True,
)


# =========================
# EXPORT
# =========================

if OUTPUT_FORMAT == "webm":
    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2500,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-auto-alt-ref", "0",
        ],
    )

elif OUTPUT_FORMAT == "mp4":
    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=2500,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
        ],
    )

else:
    raise ValueError(f"Unsupported OUTPUT_FORMAT: {OUTPUT_FORMAT}")

anim.save(OUT_FILE, writer=writer, dpi=DPI)

plt.close(fig)

print(f"Saved: {OUT_FILE}")

Saved: animations/binary_sea/binary_sea_panel.webm
